%pip install pinecone-client pymongo transformers

%pip install pinecone-client pymongo transformers

In [1]:
%pip install pinecone-client pymongo transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pinecone-client] [pymongo]n]


In [2]:
import requests
ip = requests.get("https://api.ipify.org", timeout=10).text.strip()
print(ip)

34.45.83.248


In [3]:
from pymongo import MongoClient

uri="mongodb+srv://kkforever66_db_user:<password>@cluster0.y7b1gpl.mongodb.net"
client=MongoClient(uri)
try:
    client.admin.command('ping')
    print('Pinged your deployment. You successfully connected to MongoDB!')
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [8]:
%pip uninstall pinecone-client -y

Found existing installation: pinecone-client 6.0.0
Uninstalling pinecone-client-6.0.0:
  Successfully uninstalled pinecone-client-6.0.0


In [9]:
%pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 51.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pinecone]1/2 [pinecone]


In [ ]:
from pinecone import Pinecone,ServerlessSpec

pc=Pinecone(api_key='')

In [26]:
index_name = "movies-vector-index"
dimension = 1024

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [27]:
index

Index(host='https://movies-vector-index-qd5ea7i.svc.aped-4627-b74a.pinecone.io')

In [18]:
db=client['movie_pinecone']

In [19]:
collection=db['moviecollection']

In [14]:
%pip install sentence_transformers

In [15]:
from sentence_transformers import SentenceTransformer

In [29]:
embedding_model = SentenceTransformer('thenlper/gte-large')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
import time

with collection.watch(full_document="updateLookup") as cursor:
    print("Change stream is open. Waiting for changes...")

    while True:
        change = cursor.try_next()

        if change is None:
            # 当前没有新变更，稍后继续轮询
            time.sleep(1)
            continue

        operation = change["operationType"]

        if operation in ("insert", "update", "replace"):
            document = change.get("fullDocument")
            if not document:
                print(f"{operation} 事件没有 fullDocument，跳过")
                continue

            fullplot = document.get("fullplot")
            if not fullplot:
                print("文档没有 fullplot，跳过")
                continue

            vector = embedding_model.encode(fullplot).tolist()
            vector = [float(x) for x in vector]

            index.upsert(vectors=[(str(document["_id"]), vector)])
            print(f"{operation}: 已同步 {document['_id']}")

        elif operation == "delete":
            document_id = change["documentKey"]["_id"]
            index.delete(ids=[str(document_id)])
            print(f"delete: 已删除 {document_id} 的向量")

Change stream is open. Waiting for changes...
